# MarketPulse — Data Collection

## Objective

This notebook retrieves the raw financial datasets required for the MarketPulse project from Yahoo Finance, Alpha Vantage, and SEC EDGAR.

## Data sources

* **Yahoo Finance:** Historical stock market data.
* **Alpaca:** Financial market/news data, depending on the selected API endpoints.
* **SEC EDGAR:** Company filings and/or financial statement data.

## Scope

* Companies: The approved MarketPulse ticker list.
* Market-data period: January 1, 2021 – December 31, 2025, where supported by the source.
* Frequency: Daily for historical stock prices.
* Output: Raw datasets saved under `data/raw/`.

## Workflow

1. Configure the environment.
2. Retrieve data from each source.
3. Inspect API responses and data structure.
4. Validate collection completeness.
5. Save raw datasets.
6. Summarize collection results.

**Note:** This notebook focuses on data retrieval and collection validation. Detailed cleaning, feature engineering, sentiment analysis, and risk calculations belong to subsequent project stages.


# Section 01 — Imports and configuration

In [14]:
%pip install -q yfinance transformers torch requests pandas numpy lxml datetime tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import time
import requests
import numpy as np
import pandas as pd
import yfinance as yf
import csv
import sys
import os
import json
from pathlib import Path
import datetime
from datetime import datetime
from datetime import date 


ALPACA_API_KEY = "PKFWWL3RQQFF2JPCO6KXETPBIH"
ALPACA_API_SECRET = "6QKyXtJZD4WFYtfKJt2k56jKeSBPgfugHJzUxaCxyUPa"

if not ALPACA_API_KEY or not ALPACA_API_SECRET:
    raise ValueError("Alpaca API credentials are missing.")

USER_AGENT = "basmala.e.mansour@gmail.com"   # required by SEC EDGAR

PROJECT_ROOT = Path.cwd().resolve().parents[1]  # To save data in the raw data directory, not in the notebooks directory
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True) 

# Small test version. Once everything works, uncomment the full list below.
TICKERS = ["AAPL", "JPM", "MSFT", "NVDA", "JNJ", "XOM", "KO", "AMZN", "LLY", "TSLA"]
# TICKERS = ["AAPL", "JPM", "MSFT", "NVDA", "JNJ", "XOM", "KO", "AMZN", "LLY", "TSLA"]

PRICE_START = "2021-01-01"          # start date for prices, earnings and 8-K events
NEWS_START = "2021-01-01"           # start date for news 
END = "2025-12-31"                  # end date for prices, news, earnings and 8-K events

# Section 02 — Yahoo Finance: historical stock data
Purpose: retrieve daily market data for each selected company.

In [ ]:
frames = []
for t in TICKERS:
    df = yf.download(t, start=PRICE_START, end=END, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index()
    df["ticker"] = t
    frames.append(df)

prices = pd.concat(frames, ignore_index=True)
prices.columns = [c.lower() for c in prices.columns]
prices["date"] = pd.to_datetime(prices["date"]).dt.normalize()
prices.to_csv( RAW_DATA_DIR / "yahoo_prices_raw.csv", index=False)
print("Shape:", prices.shape)
display(prices.head())
prices.info()

Shape: (12540, 7)


,date,close,high,low,open,volume,ticker
0,2021-01-04,125.632500,129.709898,123.059852,129.622529,143301900,AAPL
1,2021-01-05,127.185776,127.894478,124.681085,125.127664,97664900,AAPL
2,2021-01-06,122.904503,127.224610,122.690924,123.991812,155088000,AAPL
3,2021-01-07,127.098427,127.787708,124.127751,124.613156,109578200,AAPL
4,2021-01-08,128.195450,128.758521,126.428569,128.564348,105158200,AAPL


<class 'pandas.DataFrame'>
RangeIndex: 12540 entries, 0 to 12539
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype        
---  ------  --------------  -----        
 0   date    12540 non-null  datetime64[s]
 1   close   12540 non-null  float64      
 2   high    12540 non-null  float64      
 3   low     12540 non-null  float64      
 4   open    12540 non-null  float64      
 5   volume  12540 non-null  int64        
 6   ticker  12540 non-null  str          
dtypes: datetime64[s](1), float64(4), int64(1), str(1)
memory usage: 685.9 KB


# Section 03 — Alpaca: news and sentiment-related data
Purpose: collect the financial news data needed for MarketPulse's sentiment analysis.

In [5]:
NEWS_FILE = RAW_DATA_DIR / "alpaca_news.csv"
LOG_FILE = RAW_DATA_DIR / "alpaca_news_collection_log.csv"

NEWS_URL = "https://data.alpaca.markets/v1beta1/news"

session = requests.Session()
session.headers.update({
    "APCA-API-KEY-ID": ALPACA_API_KEY,
    "APCA-API-SECRET-KEY": ALPACA_API_SECRET,
})

print("Output directory:", RAW_DATA_DIR.resolve())
print("Tickers:", TICKERS)


Output directory: D:\DEPI\MarketPulse Graduation Project\data\raw
Tickers: ['AAPL', 'JPM', 'MSFT', 'NVDA', 'JNJ', 'XOM', 'KO', 'AMZN', 'LLY', 'TSLA']


In [4]:

def load_csv_or_empty(path):
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)
    return pd.DataFrame()


def save_news_page(articles):
    """Append new articles while keeping article IDs unique."""
    if not articles:
        return 0

    new_df = pd.DataFrame(articles)

    existing = load_csv_or_empty(NEWS_FILE)

    if not existing.empty:
        combined = pd.concat(
            [existing, new_df],
            ignore_index=True,
            sort=False,
        )
    else:
        combined = new_df

    if "id" in combined.columns:
        combined = combined.drop_duplicates(
            subset=["id"],
            keep="last",
        )

    combined.to_csv(NEWS_FILE, index=False)
    return len(new_df)


def fetch_news_page(start, end, page_token=None):
    params = {
        "start": start,
        "end": end,
        "symbols": ",".join(TICKERS),
        "limit": 50,
        "sort": "asc",
        "include_content": True,
    }

    if page_token:
        params["page_token"] = page_token

    last_error = None

    for attempt in range(6):
        try:
            response = session.get(
                NEWS_URL,
                params=params,
                timeout=(15, 90),
            )

            if response.status_code == 429 or response.status_code >= 500:
                last_error = RuntimeError(
                    f"Temporary HTTP {response.status_code}: "
                    f"{response.text[:500]}"
                )
            else:
                response.raise_for_status()
                return response.json()

        except (
            requests.exceptions.ConnectionError,
            requests.exceptions.Timeout,
            requests.exceptions.ChunkedEncodingError,
            requests.exceptions.HTTPError,
        ) as exc:
            last_error = exc

            # Don't retry ordinary client errors such as 400, 401, or 403.
            if isinstance(exc, requests.exceptions.HTTPError):
                status = exc.response.status_code
                if status not in (429,) and status < 500:
                    raise

        if attempt < 5:
            wait = min(5 * (2 ** attempt), 60)
            print(
                f"Request failed (attempt {attempt + 1}/6): "
                f"{last_error}"
            )
            print(f"Retrying in {wait} seconds...")
            time.sleep(wait)

    raise RuntimeError(
        f"Request failed after 6 attempts: {last_error}"
    )


def log_month(month, status, pages=0, articles=0, error=None):
    log = load_csv_or_empty(LOG_FILE)

    entry = pd.DataFrame([{
        "month": month,
        "status": status,
        "pages": pages,
        "articles_received": articles,
        "error": str(error) if error else "",
        "updated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    }])

    # Keep the latest status for each month.
    if not log.empty:
        log = log[log["month"] != month]

    log = pd.concat([log, entry], ignore_index=True)
    log.to_csv(LOG_FILE, index=False)


In [5]:
months = pd.period_range(
    start=NEWS_START[:7],
    end=END[:7],
    freq="M",
)

existing_log = load_csv_or_empty(LOG_FILE)

completed_months = set()

if not existing_log.empty:
    completed_months = set(
        existing_log.loc[
            existing_log["status"] == "completed",
            "month",
        ].astype(str)
    )

for month in months:
    month_key = str(month)

    if month_key in completed_months:
        print(f"Skipping completed month: {month_key}")
        continue

    start = month.start_time.strftime("%Y-%m-%dT00:00:00Z")
    end = (
        (month + 1).start_time - pd.Timedelta(seconds=1)
    ).strftime("%Y-%m-%dT%H:%M:%SZ")

    print(f"\nCollecting {month_key}...")

    page_token = None
    page_count = 0
    article_count = 0

    try:
        while True:
            result = fetch_news_page(
                start=start,
                end=end,
                page_token=page_token,
            )

            articles = result.get("news", [])
            page_count += 1
            article_count += len(articles)

            save_news_page(articles)

            print(
                f"  Page {page_count}: {len(articles)} articles"
            )

            page_token = result.get("next_page_token")

            if not page_token:
                break

            # Small pause between requests.
            time.sleep(0.25)

        log_month(
            month_key,
            status="completed",
            pages=page_count,
            articles=article_count,
        )

        print(
            f"Completed {month_key}: "
            f"{page_count} pages, "
            f"{article_count} article records received."
        )

    except Exception as exc:
        log_month(
            month_key,
            status="error",
            pages=page_count,
            articles=article_count,
            error=exc,
        )

        print(f"Failed {month_key}: {exc}")
        print("Rerun this cell later to retry incomplete months.")
        break

print("\nCollection run finished.")
print("News file:", NEWS_FILE)
print("Log file:", LOG_FILE)


Skipping completed month: 2021-01
Skipping completed month: 2021-02
Skipping completed month: 2021-03
Skipping completed month: 2021-04
Skipping completed month: 2021-05
Skipping completed month: 2021-06
Skipping completed month: 2021-07
Skipping completed month: 2021-08
Skipping completed month: 2021-09
Skipping completed month: 2021-10
Skipping completed month: 2021-11
Skipping completed month: 2021-12
Skipping completed month: 2022-01
Skipping completed month: 2022-02
Skipping completed month: 2022-03
Skipping completed month: 2022-04
Skipping completed month: 2022-05
Skipping completed month: 2022-06
Skipping completed month: 2022-07
Skipping completed month: 2022-08
Skipping completed month: 2022-09
Skipping completed month: 2022-10
Skipping completed month: 2022-11
Skipping completed month: 2022-12
Skipping completed month: 2023-01
Skipping completed month: 2023-02
Skipping completed month: 2023-03
Skipping completed month: 2023-04
Skipping completed month: 2023-05
Skipping compl

In [6]:
news = pd.read_csv(NEWS_FILE)

news["published_at"] = pd.to_datetime(
    news["created_at"],
    utc=True,
    errors="coerce",
)

print("Total unique articles:", news["id"].nunique())
print("Duplicate IDs:", news["id"].duplicated().sum())
print("Earliest article:", news["published_at"].min())
print("Latest article:", news["published_at"].max())

for col in ["headline", "summary", "content"]:
    if col in news.columns:
        print(
            f"{col} non-null: "
            f"{news[col].notna().sum()} / {len(news)}"
        )

news["month"] = news["published_at"].dt.to_period("M")

display(
    news.groupby("month")
        .size()
        .rename("article_count")
        .to_frame()
)


Total unique articles: 24121
Duplicate IDs: 0
Earliest article: 2016-09-22 13:52:48+00:00
Latest article: 2025-12-31 22:30:46+00:00
headline non-null: 24121 / 24121
summary non-null: 16413 / 24121
content non-null: 16435 / 24121


C:\Users\bosya\AppData\Local\Temp\ipykernel_17920\987012495.py:21: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  news["month"] = news["published_at"].dt.to_period("M")


,article_count
month,
2016-09,1
2017-07,1
2020-10,1
2020-11,1
2021-01,901
2021-02,925
2021-03,1057
2021-04,1508
2021-05,976


In [6]:
RECOVERY_DIR = NEWS_FILE.parent / "alpaca_news_pages"

RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

print("Recovery folder:", RECOVERY_DIR)

Recovery folder: D:\DEPI\MarketPulse Graduation Project\data\raw\alpaca_news_pages


In [7]:

def save_page_atomically(path, response_data):
    """Write one API response safely, without replacing the final CSV."""
    temp_path = path.with_suffix(".json.tmp")

    with open(temp_path, "w", encoding="utf-8") as file:
        json.dump(
            response_data,
            file,
            ensure_ascii=False,
        )
        file.flush()
        os.fsync(file.fileno())

    os.replace(temp_path, path)


def load_saved_page(path):
    """Load a previously saved response and validate its structure."""
    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    if not isinstance(data, dict) or not isinstance(data.get("news"), list):
        raise ValueError(f"Invalid saved API response: {path}")

    return data


def collect_month_safely(month_key):
    """
    Collect a month into separate JSON page files.
    Resumes from the last saved page after an interruption.
    """
    month_dir = RECOVERY_DIR / month_key
    month_dir.mkdir(parents=True, exist_ok=True)

    period = pd.Period(month_key, freq="M")

    start = period.start_time.strftime("%Y-%m-%dT00:00:00Z")
    end = (
        (period + 1).start_time - pd.Timedelta(seconds=1)
    ).strftime("%Y-%m-%dT%H:%M:%SZ")

    # Find the longest uninterrupted sequence of saved pages.
    page_number = 1
    while (month_dir / f"page_{page_number:05d}.json").exists():
        page_number += 1

    last_saved_page = page_number - 1

    if last_saved_page:
        previous_path = month_dir / f"page_{last_saved_page:05d}.json"
        previous_response = load_saved_page(previous_path)
        page_token = previous_response.get("next_page_token")

        if not page_token:
            print(f"{month_key}: already fully collected.")
            return

        page_number = last_saved_page + 1
    else:
        page_token = None

    print(f"\nCollecting/resuming {month_key} from page {page_number}...")

    while True:
        response_data = fetch_news_page(
            start=start,
            end=end,
            page_token=page_token,
        )

        articles = response_data.get("news", [])

        page_path = month_dir / f"page_{page_number:05d}.json"
        save_page_atomically(page_path, response_data)

        print(
            f"  Saved page {page_number}: "
            f"{len(articles)} articles"
        )

        page_token = response_data.get("next_page_token")

        if not page_token:
            print(f"Completed {month_key}.")
            break

        page_number += 1
        time.sleep(0.25)


In [8]:

current_news = pd.read_csv(NEWS_FILE, usecols=["created_at"])

current_news["created_at"] = pd.to_datetime(
    current_news["created_at"],
    errors="coerce",
    utc=True,
)

current_news["month"] = current_news["created_at"].dt.strftime("%Y-%m")
current_counts = current_news["month"].value_counts()

log = pd.read_csv(LOG_FILE)
log["month"] = log["month"].astype(str)

log["rows_in_csv"] = (
    log["month"].map(current_counts).fillna(0).astype(int)
)

log["articles_received"] = pd.to_numeric(
    log["articles_received"],
    errors="coerce",
).fillna(0).astype(int)

# Recover months where the saved CSV has fewer rows than
# the collector reported receiving.
recovery_plan = log.loc[
    log["articles_received"] > log["rows_in_csv"],
    ["month", "articles_received", "rows_in_csv"],
].copy()

recovery_plan["missing_vs_log"] = (
    recovery_plan["articles_received"] - recovery_plan["rows_in_csv"]
)

display(recovery_plan.sort_values("month"))
print("Months to inspect:", len(recovery_plan))


,month,articles_received,rows_in_csv,missing_vs_log
7,2021-08,1250,1242,8
11,2021-12,1152,1144,8
12,2022-01,1214,1212,2
13,2022-02,1048,1047,1
15,2022-04,1653,230,1423
16,2022-05,1273,0,1273
17,2022-06,1332,0,1332
18,2022-07,1433,0,1433
19,2022-08,1244,0,1244
20,2022-09,1265,0,1265


Months to inspect: 46


In [9]:

months_to_recover = recovery_plan["month"].tolist()

for month_key in months_to_recover:
    collect_month_safely(month_key)

print("\nRecovery collection finished.")
print("Saved pages are in:", RECOVERY_DIR)


2021-08: already fully collected.
2021-12: already fully collected.
2022-01: already fully collected.
2022-02: already fully collected.
2022-04: already fully collected.
2022-05: already fully collected.
2022-06: already fully collected.
2022-07: already fully collected.
2022-08: already fully collected.
2022-09: already fully collected.
2022-10: already fully collected.
2022-11: already fully collected.
2022-12: already fully collected.
2023-01: already fully collected.
2023-02: already fully collected.
2023-03: already fully collected.
2023-04: already fully collected.
2023-05: already fully collected.
2023-06: already fully collected.
2023-07: already fully collected.
2023-08: already fully collected.
2023-09: already fully collected.
2023-10: already fully collected.
2023-11: already fully collected.
2023-12: already fully collected.
2024-01: already fully collected.
2024-02: already fully collected.
2024-03: already fully collected.
2024-04: already fully collected.
2024-05: alrea

In [10]:
# Increase the CSV field-size limit safely.
limit = sys.maxsize

while True:
    try:
        csv.field_size_limit(limit)
        break
    except OverflowError:
        limit //= 10

print("CSV field-size limit:", csv.field_size_limit())

CSV field-size limit: 9223372036854775807


In [ ]:
RECOVERED_FILE = RAW_DATA_DIR / "alpaca_news_recovered.csv"
check = pd.read_csv(
    RECOVERED_FILE,
    usecols=["id", "created_at"],
)

check["created_at"] = pd.to_datetime(
    check["created_at"],
    errors="coerce",
    utc=True,
)

print("Total rows:", len(check))
print("Unique IDs:", check["id"].nunique())
print("Duplicate IDs:", check["id"].duplicated().sum())
print("Earliest date:", check["created_at"].min())
print("Latest date:", check["created_at"].max())

print("\nMonthly article counts:")
print(
    check.groupby(check["created_at"].dt.strftime("%Y-%m"))
    .size()
    .sort_index()
    .to_string()
)

Total rows: 79565
Unique IDs: 79565
Duplicate IDs: 0
Earliest date: 2014-11-07 14:17:17+00:00
Latest date: 2025-12-31 22:30:46+00:00

Monthly article counts:
created_at
2014-11       1
2016-09       1
2017-07       1
2018-03       1
2019-05       2
2019-09       1
2019-10       1
2020-02       1
2020-04       1
2020-05       1
2020-09       1
2020-10       3
2020-11       1
2021-01     902
2021-02     926
2021-03    1058
2021-04    1509
2021-05     976
2021-06    1273
2021-07    1493
2021-08    1245
2021-09    1313
2021-10    1407
2021-11    1178
2021-12    1144
2022-01    1212
2022-02    1049
2022-03    1360
2022-04    1657
2022-05    1275
2022-06    1328
2022-07    1440
2022-08    1244
2022-09    1267
2022-10    1584
2022-11    1247
2022-12    1108
2023-01    1215
2023-02     974
2023-03    1241
2023-04    1438
2023-05    1530
2023-06    1468
2023-07    1558
2023-08    1519
2023-09    1459
2023-10    1617
2023-11    1418
2023-12    1212
2024-01    1448
2024-02    1268
2024-03    1281

In [ ]:
FINAL_FILE = RAW_DATA_DIR / "alpaca_news_2021_2025_validated.csv"
check = pd.read_csv(
    FINAL_FILE,
    usecols=["id", "created_at"],
)

check["created_at"] = pd.to_datetime(
    check["created_at"],
    utc=True,
    errors="coerce",
)

monthly = check.groupby(
    check["created_at"].dt.strftime("%Y-%m")
).size()

expected_months = pd.period_range(
    "2021-01",
    "2025-12",
    freq="M",
).astype(str)

monthly = monthly.reindex(expected_months, fill_value=0)

print("Total rows:", len(check))
print("Unique IDs:", check["id"].nunique())
print("Months covered:", len(monthly))
print("Months with zero articles:", int((monthly == 0).sum()))
print(monthly.to_string())

assert len(check) == check["id"].nunique(), "Duplicate IDs detected!"
assert (monthly > 0).all(), "At least one month is empty!"

Total rows: 79549
Unique IDs: 79549
Months covered: 60
Months with zero articles: 0
2021-01     902
2021-02     926
2021-03    1058
2021-04    1509
2021-05     976
2021-06    1273
2021-07    1493
2021-08    1245
2021-09    1313
2021-10    1407
2021-11    1178
2021-12    1144
2022-01    1212
2022-02    1049
2022-03    1360
2022-04    1657
2022-05    1275
2022-06    1328
2022-07    1440
2022-08    1244
2022-09    1267
2022-10    1584
2022-11    1247
2022-12    1108
2023-01    1215
2023-02     974
2023-03    1241
2023-04    1438
2023-05    1530
2023-06    1468
2023-07    1558
2023-08    1519
2023-09    1459
2023-10    1617
2023-11    1418
2023-12    1212
2024-01    1448
2024-02    1268
2024-03    1281
2024-04    1502
2024-05    1539
2024-06    1395
2024-07    1605
2024-08    1208
2024-09    1182
2024-10    1432
2024-11    1193
2024-12    1082
2025-01    1390
2025-02    1110
2025-03    1176
2025-04    1458
2025-05    1387
2025-06    1255
2025-07    1628
2025-08    1306
2025-09    1421
2025

# Section 04 — Yahoo Finance: Earnings Events
Purpose: collect historical earnings announcements and EPS results to analyze how companies' earnings events relate to stock-price reactions.

One row per earnings announcement: the date, the expected EPS, the actual EPS and the surprise.
`reaction_date` is the first trading day when the market could react. If the company announced
after 4 PM (market close), the reaction happens the next business day.

In [27]:
ev_frames = []
for t in TICKERS:
    try:
        ed = yf.Ticker(t).get_earnings_dates(limit=100)
    except Exception as e:
        print(t, "earnings error:", e)
        continue
    if ed is None or ed.empty:
        print(t, "no earnings data returned")
        continue
    ed = ed.reset_index()
    ed = ed.rename(columns={
        "Earnings Date": "announce_datetime",
        "EPS Estimate": "eps_estimate",
        "Reported EPS": "eps_actual",
        "Surprise(%)": "surprise_pct",
    })
    ed["ticker"] = t
    ev_frames.append(ed)

earnings = pd.concat(ev_frames, ignore_index=True)

dt = pd.to_datetime(earnings["announce_datetime"])
if dt.dt.tz is not None:
    dt = dt.dt.tz_localize(None)          # keep the local (New York) clock time
earnings["announce_datetime"] = dt
earnings["announce_date"] = dt.dt.normalize()
earnings["after_close"] = dt.dt.hour >= 16
earnings["reaction_date"] = earnings["announce_date"]
late = earnings["after_close"]
earnings.loc[late, "reaction_date"] = earnings.loc[late, "announce_date"] + pd.offsets.BDay(1)

# keep only announcements that already happened and are inside our period
earnings["announce_date"] = pd.to_datetime(
    earnings["announce_date"]
)

earnings = earnings[
    earnings["eps_actual"].notna()
    & (earnings["announce_date"] >= pd.Timestamp(PRICE_START))
    & (earnings["announce_date"] < pd.Timestamp("2026-01-01"))
].copy()
earnings = earnings[["ticker", "announce_datetime", "announce_date", "after_close", "reaction_date",
                     "eps_estimate", "eps_actual", "surprise_pct"]].sort_values(["ticker", "announce_date"])
earnings.to_csv(RAW_DATA_DIR / "earnings_events.csv", index=False)
print(earnings.groupby("ticker").size())
earnings

ticker
AAPL    20
AMZN    20
JNJ     20
JPM     20
KO      20
LLY     20
MSFT    20
NVDA    20
TSLA    20
XOM     20
dtype: int64


,ticker,announce_datetime,announce_date,after_close,reaction_date,eps_estimate,eps_actual,surprise_pct
23,AAPL,2021-01-27 16:00:00,2021-01-27,True,2021-01-28,1.42,1.68,18.67
22,AAPL,2021-04-28 16:00:00,2021-04-28,True,2021-04-29,0.98,1.40,42.16
21,AAPL,2021-07-27 16:00:00,2021-07-27,True,2021-07-28,1.01,1.30,28.18
20,AAPL,2021-10-28 16:00:00,2021-10-28,True,2021-10-29,1.24,1.24,0.17
19,AAPL,2022-01-27 16:00:00,2022-01-27,True,2022-01-28,1.89,2.10,11.20
...,...,...,...,...,...,...,...,...
508,XOM,2024-11-01 06:00:00,2024-11-01,False,2024-11-01,1.87,1.92,2.60
507,XOM,2025-01-31 06:00:00,2025-01-31,False,2025-01-31,1.55,1.67,7.87
506,XOM,2025-05-02 06:00:00,2025-05-02,False,2025-05-02,1.74,1.76,1.16
505,XOM,2025-08-01 06:00:00,2025-08-01,False,2025-08-01,1.56,1.64,4.93


# Section 05 — SEC EDGAR: Company Fundamentals
Purpose: collect company financial and fundamental metrics to provide business-performance context for MarketPulse's stock and financial news analysis.

Important: each value is stored with its `filing_date` (the day the public could see it),
so we never use numbers before they were published.

Note: income items (revenue, net income, EPS) are kept only for single quarters (from 10-Q filings).
The fourth quarter is reported inside the annual 10-K, so it is not included here.

In [3]:
HEADERS = {"User-Agent": USER_AGENT}

tick_map = requests.get("https://www.sec.gov/files/company_tickers.json", headers=HEADERS, timeout=60).json()
cik_by_ticker = {v["ticker"]: int(v["cik_str"]) for v in tick_map.values()}

# metric name -> (unit, list of possible XBRL tags; companies sometimes switch tags over the years)
METRICS = {
    "revenue": ("USD", ["Revenues",
                        "RevenueFromContractWithCustomerExcludingAssessedTax",
                        "SalesRevenueNet"]),
    "net_income": ("USD", ["NetIncomeLoss"]),
    "eps_diluted": ("USD/shares", ["EarningsPerShareDiluted"]),
    "total_assets": ("USD", ["Assets"]),
    "equity": ("USD", ["StockholdersEquity"]),
}

cik_by_ticker["XOM"] = 34088
rows = []
for t in TICKERS:
    cik = cik_by_ticker.get(t)
    if cik is None:
        print(t, "not found in the SEC ticker list")
        continue
    try:
        r = requests.get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json",
                         headers=HEADERS, timeout=60)
        r.raise_for_status()
        facts = r.json()["facts"].get("us-gaap", {})
    except Exception as e:
        print(t, "EDGAR error:", e)
        continue

    for metric, (unit, tags) in METRICS.items():
        for tag in tags:
            for item in facts.get(tag, {}).get("units", {}).get(unit, []):
                if item.get("form") not in ("10-Q", "10-K"):
                    continue
                rows.append({
                    "ticker": t, "metric": metric,
                    "period_start": item.get("start"), "period_end": item["end"],
                    "value": item["val"], "form": item["form"], "filing_date": item["filed"],
                })
    print(t, "done")
    time.sleep(0.3)   # stay far below the SEC limit of 10 requests per second

fund_long = pd.DataFrame(rows)
for c in ["period_start", "period_end", "filing_date"]:
    fund_long[c] = pd.to_datetime(fund_long[c])

# keep single-quarter flows (about 90 days) and point-in-time balance sheet items (no start date)
days = (fund_long["period_end"] - fund_long["period_start"]).dt.days
keep = fund_long["period_start"].isna() | days.between(80, 100)
fund_long = fund_long[keep]

fund_long["period_end"] = pd.to_datetime(fund_long["period_end"], errors="coerce")
fund_long["filing_date"] = pd.to_datetime(fund_long["filing_date"], errors="coerce")


# Sort by filing date and keep the earliest published version of each fact.
fund_long = (
    fund_long.sort_values("filing_date")
    .drop_duplicates(
        subset=["ticker", "metric", "period_start", "period_end"],
        keep="first"
    )
)

# Build one table of values.
wide = fund_long.pivot_table(
    index=["ticker", "period_end"],
    columns="metric",
    values="value",
    aggfunc="first"
).reset_index()

# Build a separate table of publication dates for each metric.
publication_dates = fund_long.pivot_table(
    index=["ticker", "period_end"],
    columns="metric",
    values="filing_date",
    aggfunc="min"
).reset_index()

# Give the publication-date columns distinct names.
publication_dates = publication_dates.rename(
    columns={
        metric: f"{metric}_filing_date"
        for metric in METRICS
        if metric in publication_dates.columns
    }
)

fundamentals = wide.merge(
    publication_dates,
    on=["ticker", "period_end"],
    how="left"
)


# Step 1: Keep records within the project's target period
fundamentals = fundamentals[
    (fundamentals["period_end"] >= PRICE_START)
    & (fundamentals["period_end"] < END)
].copy()

# Step 2: Exclude financial values filed on or after the end date
for metric in METRICS:
    date_col = f"{metric}_filing_date"

    if date_col in fundamentals.columns:
        fundamentals.loc[
            fundamentals[date_col] >= END, metric
        ] = np.nan

# Step 3: Calculate profit margin
fundamentals["profit_margin"] = (
    fundamentals["net_income"] / fundamentals["revenue"]
)

# Step 4: Clean up the table's column index name
fundamentals.columns.name = None

# Step 5: Sort and reset the row numbers
fundamentals = (
    fundamentals
    .sort_values(["ticker", "period_end"])
    .reset_index(drop=True)
)

# Step 6: Save the final dataset
fundamentals.to_csv(RAW_DATA_DIR / "company_fundamentals.csv", index=False)

print("Final fundamentals shape:", fundamentals.shape)
fundamentals


AAPL done
JPM done
MSFT done
NVDA done
JNJ done
XOM done
KO done
AMZN done
LLY done
TSLA done
Final fundamentals shape: (194, 13)


,ticker,period_end,eps_diluted,equity,net_income,revenue,total_assets,eps_diluted_filing_date,equity_filing_date,net_income_filing_date,revenue_filing_date,total_assets_filing_date,profit_margin
0,AAPL,2021-03-27,1.40,6.917800e+10,2.363000e+10,8.958400e+10,3.371580e+11,2021-04-29,2021-04-29,2021-04-29,2021-04-29,2021-04-29,0.263775
1,AAPL,2021-06-26,1.30,6.428000e+10,2.174400e+10,8.143400e+10,3.298400e+11,2021-07-28,2021-07-28,2021-07-28,2021-07-28,2021-07-28,0.267014
2,AAPL,2021-09-25,NaN,6.309000e+10,NaN,NaN,3.510020e+11,NaT,2021-10-29,NaT,NaT,2021-10-29,NaN
3,AAPL,2021-12-25,2.10,7.193200e+10,3.463000e+10,1.239450e+11,3.811910e+11,2022-01-28,2022-01-28,2022-01-28,2022-01-28,2022-01-28,0.279398
4,AAPL,2022-03-26,1.52,6.739900e+10,2.501000e+10,9.727800e+10,3.506620e+11,2022-04-29,2022-04-29,2022-04-29,2022-04-29,2022-04-29,0.257098
...,...,...,...,...,...,...,...,...,...,...,...,...,...
189,XOM,2024-09-30,1.92,2.685920e+11,8.610000e+09,9.001600e+10,4.619160e+11,2024-11-04,2024-11-04,2024-11-04,2024-11-04,2024-11-04,0.095650
190,XOM,2024-12-31,NaN,2.637050e+11,NaN,NaN,4.534750e+11,NaT,2025-02-19,NaT,NaT,2025-02-19,NaN
191,XOM,2025-03-31,1.76,2.627200e+11,7.713000e+09,8.313000e+10,4.519080e+11,2025-05-05,2025-05-05,2025-05-05,2025-05-05,2025-05-05,0.092782
192,XOM,2025-06-30,1.64,2.625930e+11,7.082000e+09,8.150600e+10,4.475970e+11,2025-08-04,2025-08-04,2025-08-04,2025-08-04,2025-08-04,0.086889


# Section 06 — SEC EDGAR: Official Company Announcements (8-K Filings)
Purpose: collect official company filings and announcements to identify significant corporate events and examine their relationship with financial news sentiment and stock-market reactions.

An 8-K is a report a company must file when something important happens
(results, executive changes, acquisitions...). We keep the date and the event type

In [8]:
ITEM_NAMES = {
    "1.01": "Entry into a material agreement",
    "1.02": "Termination of a material agreement",
    "2.01": "Completion of acquisition or disposition",
    "2.02": "Results of operations (earnings)",
    "2.03": "New financial obligation",
    "2.05": "Exit or restructuring costs",
    "3.02": "Unregistered sale of equity",
    "5.02": "Executive or director change",
    "5.07": "Shareholder vote results",
    "7.01": "Regulation FD disclosure",
    "8.01": "Other events",
    "9.01": "Financial statements and exhibits",
}

cik_by_ticker["XOM"] = 34088
rows = []

start_date = pd.Timestamp(PRICE_START)
end_date = pd.Timestamp(END)

for t in TICKERS:
    cik = cik_by_ticker.get(t)

    if cik is None:
        print(t, "CIK not found")
        continue

    base_url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"

    try:
        response = requests.get(
            base_url, headers=HEADERS, timeout=60
        )
        response.raise_for_status()
        company_data = response.json()

        print(t, "|", company_data.get("name"), "| CIK:", cik)

        filing_data = company_data["filings"]
        recent = pd.DataFrame(filing_data["recent"])
        filing_frames = [recent]

        # Find historical files overlapping our target period
        for file_info in filing_data.get("files", []):
            filing_from = pd.to_datetime(
                file_info.get("filingFrom"), errors="coerce"
            )
            filing_to = pd.to_datetime(
                file_info.get("filingTo"), errors="coerce"
            )

            if pd.isna(filing_from) or pd.isna(filing_to):
                continue

            # Skip files entirely outside the target period
            if filing_to < start_date or filing_from >= end_date:
                continue

            historical_url = (
                "https://data.sec.gov/submissions/"
                + file_info["name"]
            )

            hist_response = requests.get(
                historical_url, headers=HEADERS, timeout=60
            )
            hist_response.raise_for_status()

            filing_frames.append(pd.DataFrame(hist_response.json()))
            time.sleep(0.3)

        # Combine recent and historical filing records
        all_filings = pd.concat(
            filing_frames, ignore_index=True
        )

        # Filter for 8-K filings in the target period
        all_filings["filingDate"] = pd.to_datetime(
            all_filings["filingDate"], errors="coerce"
        )

        selected = all_filings[
            (all_filings["form"] == "8-K")
            & (all_filings["filingDate"] >= start_date)
            & (all_filings["filingDate"] < end_date)
        ].copy()

        for _, x in selected.iterrows():
            codes = [
                c.strip()
                for c in str(x.get("items", "")).split(",")
                if c.strip() and c.strip().lower() != "nan"
            ]

            rows.append({
                "ticker": t,
                "filing_date": x["filingDate"],
                "form": x["form"],
                "accession_number": x["accessionNumber"],
                "items": ",".join(codes),
                "items_description": "; ".join(
                    ITEM_NAMES.get(c, "Other") for c in codes
                ),
            })

        print(t, "8-K filings collected:", len(selected))

    except Exception as e:
        print(t, "EDGAR error:", e)

    time.sleep(0.3)

# Build and save the final dataset
events_8k = pd.DataFrame(
    rows,
    columns=[
        "ticker", "filing_date", "form",
        "items", "items_description" ,"accession_number"
    ]
)

events_8k["filing_date"] = pd.to_datetime(
    events_8k["filing_date"], errors="coerce"
)

events_8k = (
    events_8k
    .drop_duplicates(subset=["accession_number"])
    .sort_values(["ticker", "filing_date"])
    .reset_index(drop=True)
)

print("\nEarliest 8-K date per ticker:")
print(events_8k.groupby("ticker")["filing_date"].min())

print("\nNumber of 8-K filings per ticker:")
print(events_8k.groupby("ticker").size())

print("\nFinal dataset shape:", events_8k.shape)

# Save the dataset
events_8k.to_csv(RAW_DATA_DIR / "events_8k.csv", index=False)

display(events_8k.head())


AAPL | Apple Inc. | CIK: 320193
AAPL 8-K filings collected: 42
JPM | JPMORGAN CHASE & CO | CIK: 19617
JPM 8-K filings collected: 141
MSFT | MICROSOFT CORP | CIK: 789019
MSFT 8-K filings collected: 47
NVDA | NVIDIA CORP | CIK: 1045810
NVDA 8-K filings collected: 46
JNJ | JOHNSON & JOHNSON | CIK: 200406
JNJ 8-K filings collected: 60
XOM | EXXON MOBIL CORP | CIK: 34088
XOM 8-K filings collected: 80
KO | COCA COLA CO | CIK: 21344
KO 8-K filings collected: 66
AMZN | AMAZON COM INC | CIK: 1018724
AMZN 8-K filings collected: 47
LLY | ELI LILLY & Co | CIK: 59478
LLY 8-K filings collected: 52
TSLA | Tesla, Inc. | CIK: 1318605
TSLA 8-K filings collected: 63

Earliest 8-K date per ticker:
ticker
AAPL   2021-01-05
AMZN   2021-01-27
JNJ    2021-01-26
JPM    2021-01-15
KO     2021-02-10
LLY    2021-01-14
MSFT   2021-01-26
NVDA   2021-02-24
TSLA   2021-01-04
XOM    2021-02-02
Name: filing_date, dtype: datetime64[us]

Number of 8-K filings per ticker:
ticker
AAPL     42
AMZN     47
JNJ      60
JPM    

,ticker,filing_date,form,items,items_description,accession_number
0,AAPL,2021-01-05,8-K,5.02,Executive or director change,0001193125-21-001982
1,AAPL,2021-01-27,8-K,"2.02,9.01",Results of operations (earnings); Financial st...,0000320193-21-000009
2,AAPL,2021-02-08,8-K,"8.01,9.01",Other events; Financial statements and exhibits,0001193125-21-032394
3,AAPL,2021-02-24,8-K,5.07,Shareholder vote results,0001193125-21-054710
4,AAPL,2021-04-28,8-K,"2.02,9.01",Results of operations (earnings); Financial st...,0000320193-21-000055



# Conclusion: MarketPulse Data Collection

## Overview

This notebook establishes the data collection stage of the MarketPulse graduation project. Its purpose is to gather and organize financial market data, financial news, company fundamentals, and corporate announcements for the selected companies over the period from January 2021 to December 2025.

The collected datasets provide the foundation for investigating relationships between financial news sentiment, corporate events, and subsequent stock market behavior.

## Data Sources and Collected Datasets

The notebook brings together data from multiple sources, including:

- **Stock Market Data:** Historical stock prices and related market information used to examine stock returns and volatility.
- **Financial News Data:** News articles associated with the selected companies, providing textual data for sentiment analysis.
- **Company Fundamentals:** Financial indicators collected from SEC EDGAR Company Facts, including revenue, net income, diluted earnings per share, total assets, equity, and derived profit margins.
- **SEC Company Announcements:** Form 8-K filing metadata, including filing dates, reported item codes, descriptions, and accession numbers, to identify significant corporate events.

Each dataset serves a different purpose and will contribute to the project's overall analysis.

## Data Validation and Quality Assurance

Data collection was accompanied by checks intended to improve reliability and consistency, including:

- Verifying company ticker mappings and SEC Central Index Key (CIK) identifiers.
- Applying the project's target date range where appropriate.
- Checking missing values, duplicate records, and date coverage.
- Organizing financial indicators into structured tables.
- Preserving filing dates and identifiers to support traceability and prevent information from being used before it became publicly available.

Although these checks improve data quality, each dataset still requires appropriate validation for completeness, consistency, and suitability for the intended analysis.

## Outputs

The collection stage produces structured datasets that can be saved as CSV files and reused throughout the project. Examples include:

- `alpaca_news_2021_2025_validated.csv`
- `fundamentals.csv`
- `events_8k.csv`

Additional market-data, earnings, and intermediate files may be generated by their respective collection sections.

## Limitations

Data availability, API restrictions, historical coverage, and differences in reporting formats may affect the completeness of individual datasets. Some financial metrics may be unavailable for certain reporting periods, while the SEC 8-K dataset currently contains filing metadata rather than the full announcement text.

The collected data should therefore be reviewed before integration. In particular, publication dates must be respected when aligning financial information with news and stock movements to avoid look-ahead bias.

## Next Steps

The next stage of MarketPulse is to prepare the collected datasets for analysis by:

1. Completing dataset-specific quality checks and documenting any limitations.
2. Standardizing dates, company identifiers, and relevant data types.
3. Integrating financial news, stock market data, fundamentals, earnings information, and corporate events using appropriate keys and timestamps.
4. Engineering analytical features, including sentiment scores, stock returns, and volatility measures.
5. Preparing the resulting analytical dataset for exploratory data analysis, visualization, and subsequent modeling or statistical analysis.

## Final Remarks

The data collection stage establishes the foundation for MarketPulse by bringing together multiple sources of financial and textual information. The quality and traceability of these inputs are essential for producing meaningful, reproducible, and reliable findings about the relationship between financial news, corporate events, and stock market behavior.
